# V20: one larger native classifier batch

**FINAL: serviceERROR after CUDAOOMduring later-source inference; original engineering checks pass, runtime fails, full quality is unmeasured.**

One12288padded-token/max32-row configuration; unchanged checkpoint, native decoder/head, formatter, tokens, pooling and equal-weight/T1Gemma+mirror ensemble. H65passed numerical checks but projected9.66hours. Its quality was not evaluated. The current official best remains0.82987/rank23until a fresh official result proves otherwise. Original votes overlap training; later votes are inspected/domain-shift robustness data, not competition CV or a hidden-score forecast. [Protocol](../docs/SPAN_020_PLAN.md).

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
REPO=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
PUB=REPO/'results/span_020'
def read(name):return json.loads((PUB/name).read_text())
p=read('span_plan.json');b=read('batch_candidate_bundle.json');c=read('batch_candidate_cell_rehearsals.json')
display(pd.DataFrame([{'configuration':'H65measured','padded_tokens':8192,'max_rows':16},
 {'configuration':'H66single candidate','padded_tokens':p['padded_tokens_cap'],'max_rows':p['max_rows_per_batch']}]))
print('Serialized CPU checks passed:',c['passed'],'; actual new-shape GPU exercised at preparation:',c['real_model_gpu_exercised'])
print('Public cap seconds:',p['public_cap_seconds'],'; inherited deadline:',p['deadline'])
print('Original resource period reset:',p['resource_period_reset'])
print('Frozen notebook SHA256:',b['notebook_sha256'])
q=read('quality_preparation.json')
print('Source/control reverse comparator rows:',{k:v['selected_rows'] for k,v in q['routes'].items()})
print('No human labels used to prepare routes:',not q['uses_winner_labels'])
job=PUB/'jobs/batch_candidate'
for name in ['status.json','controller_state.json','progress.json','source_verification.json']:
 if (job/name).exists():
  r=json.loads((job/name).read_text())
  if name=='progress.json':r={'observed_at':r['observed_at'],'latest':r['latest']}
  print(name,r)

,configuration,padded_tokens,max_rows
0,H65measured,8192,16
1,H66single candidate,12288,32


Serialized CPU checks passed: True ; actual new-shape GPU exercised at preparation: False
Public cap seconds: 5400 ; inherited deadline: 2026-11-10T08:37:37.343197+00:00
Original resource period reset: False
Frozen notebook SHA256: 954395768999aeb17c3f6ea0ca75cbc2f08f97d1c5882958cef3a97003c1bafe
Source/control reverse comparator rows: {'source': 175, 'control': 42}
No human labels used to prepare routes: True
status.json {'status': 'ERROR', 'kernel_version': 'arseniytrushin/llm-preference-span020-batch-candidate/1', 'observed_at': '2026-11-08T13:15:20.553927+00:00'}
controller_state.json {'stage': 'failed', 'at': '2026-11-08T13:15:31.758256+00:00', 'pid': 4140906, 'role': 'batch_candidate', 'status': 'ERROR', 'kernel_version': 'arseniytrushin/llm-preference-span020-batch-candidate/1', 'charged_seconds': 2012.0999999999985, 'files': 31}
progress.json {'observed_at': '2026-11-08T13:14:34.915273+00:00', 'latest': {'stage': 'mirror_progress', 'gpu': 0, 'rows': 487, 'seconds': 430.359922237

In [2]:
if not (PUB/'candidate_qualification.json').exists():
 print('No full qualification or model quality result at this snapshot.')
 if (PUB/'partial_engineering_audit.json').exists():
  e=read('partial_engineering_audit.json')
  print('Actual service status:',e['service_status'],'; failure:',e['failure_type'],'in',e['failed_stage'])
  print('Recovered original-data engineering checks passed:',e['original_engineering_checks_passed'])
  print('Native/repeated rawlogit controls:',e['native_controls'],e['repeated_logits'])
  print('Retained canonical probability max difference:',e['retained_canonical_probability_max_abs'])
  print('Full source prediction rows retrieved:',e['source_prediction_rows_retrieved'])
  print('Actual source reverse comparator complete:',e['actual_source_reverse_complete'])
  print('Conservative complete-original profile hours:',e['runtime']['projected_hidden_seconds']/3600)
  print('Human quality votes read:',e['human_votes_read'],'; official eligible:',e['official_submission_eligible'])
 if (PUB/'runtime_live_observation.json').exists():
  r=read('runtime_live_observation.json')
  print('Actual mirror512seconds:',r['mirror_profile_seconds'])
  print('Throughput ratio versus H65:',r['measured_mirror_throughput_ratio'])
  print('Declared hidden runtime lower bound hours:',r['lower_bound_hours'])
  print('Startup/warmup/tokenization/other costs omitted:',r['startup_warmup_tokenization_and_other_costs_omitted'])
else:
 d=read('candidate_qualification.json');r=d['runtime']
 print('Engineering checks passed:',d['engineering_checks_passed'])
 print('Runtime gate passed:',r['runtime_gate_passed'],'; conservative hours:',r['projected_hidden_seconds']/3600)
 print('Native numerical controls:',d['native_controls'],'; repeated rawlogits:',d['repeated_logits'])
 print('Old/new shape identity claimed:',d['old_new_shape_identity_claimed'])
 display(pd.DataFrame(d['old_new_batch_shape_drift']).T)
 print('Quality votes read:',d['human_votes_read'],'; official eligible:',d['official_submission_eligible'])
 if d['quality_evaluated']:
  rows=[]
  for name,e in d['evaluations'].items():
   rows.append({'cohort':name,'baseline_loss':e['baseline']['log_loss'],'candidate_loss':e['candidate']['log_loss'],
    'loss_reduction':e['loss_reduction'],'group95lower':e['paired_group95'][0],'group95upper':e['paired_group95'][1],
    'accuracy':e['candidate']['accuracy'],'brier':e['candidate']['brier']})
  display(pd.DataFrame(rows))
  fig,ax=plt.subplots(figsize=(8,3.5));names=list(d['evaluations'])
  ax.bar([i-.18 for i in range(len(names))],[d['evaluations'][n]['baseline']['log_loss'] for n in names],.36,label='Retained H60')
  ax.bar([i+.18 for i in range(len(names))],[d['evaluations'][n]['candidate']['log_loss'] for n in names],.36,label='Fixed candidate')
  ax.set_xticks(range(len(names)),names);ax.set_ylabel('Log loss; inspected populations');ax.legend();fig.tight_layout();plt.show()
 else:print('The engineering stop prevents a quality conclusion.')

No full qualification or model quality result at this snapshot.
Actual service status: ERROR ; failure: torch.cuda.OutOfMemoryError in later_source1024
Recovered original-data engineering checks passed: True
Native/repeated rawlogit controls: {'native_parallel_logits_max_abs': 0.0, 'native_parallel_probability_max_abs': 0.0, 'repeated_probability_max_abs': 0.0, 'rows': 12, 'token_fingerprint': '37a129d81304bc080c6b02d1a7b12603776d9aafec387805f0bfec762f9a170d', 'tolerance': 1e-05, 'passed': True} {'passed': True, 'raw_logit_max_abs': 0.0, 'tolerance': 1e-05, 'rows': 12}
Retained canonical probability max difference: 0.0
Full source prediction rows retrieved: 0
Actual source reverse comparator complete: False
Conservative complete-original profile hours: 9.136799818730532
Human quality votes read: False ; official eligible: False
Actual mirror512seconds: 267.8156185330001
Throughput ratio versus H65: 1.1148571547114967
Declared hidden runtime lower bound hours: 9.019576478320316
Startup/